<a href="https://colab.research.google.com/github/jairaj023/Internship-Practice/blob/main/Day_15Task.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

IMPORT LIBRARIES

In [26]:
import re
import spacy
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (precision_score, recall_score, f1_score, accuracy_score, confusion_matrix)

LOAD DATASET

In [7]:
from google.colab import files
upload = files.upload()

Saving job_skill_extraction_results.csv to job_skill_extraction_results.csv


In [8]:
df = pd.read_csv("job_skill_extraction_results.csv")

In [9]:
print("Dataset shape:", df.shape)

Dataset shape: (10000, 10)


In [10]:
print(df.head())

     job_id         job_title  company              location  \
0  JOB00001  Python Developer  Infosys                Remote   
1  JOB00002    Data Scientist  Infosys  Hyderabad, Telangana   
2  JOB00003     Data Engineer      IBM  Bengaluru, Karnataka   
3  JOB00004  Business Analyst   Google  Hyderabad, Telangana   
4  JOB00005  Business Analyst  Mphasis          Delhi, India   

                                     job_description  experience education  \
0  We are looking for a Python Developer to join ...   1-3 years       MCA   
1  We are looking for a Data Scientist to join ou...   0-2 years       MCA   
2  We are looking for a Data Engineer to join our...  8-12 years       BCA   
3  We are looking for a Business Analyst to join ...   1-3 years    M.Tech   
4  We are looking for a Business Analyst to join ...  8-12 years    M.Tech   

       salary    job_type                                   extracted_skills  
0  ₹10-16 LPA   Part-time                         ['Flask', 'SQL', 

CHECK COLUMN NAMES

In [11]:
print("\nColumns:")
print(df.columns.tolist())


Columns:
['job_id', 'job_title', 'company', 'location', 'job_description', 'experience', 'education', 'salary', 'job_type', 'extracted_skills']


CHANGE THESE COLUMN NAMES

In [12]:
TEXT_COLUMN = "job_description"
ACTUAL_COLUMN = "extracted_skills"

CLEAN DATA

In [13]:
df[TEXT_COLUMN] = df[TEXT_COLUMN].fillna("").astype(str)
df[ACTUAL_COLUMN] = df[ACTUAL_COLUMN].fillna("").astype(str)

SKILL DICTIONARY

In [15]:
skills = [
    "python",
    "java",
    "c++",
    "javascript",
    "sql",
    "mysql",
    "postgresql",
    "mongodb",
    "excel",
    "power bi",
    "tableau",
    "pandas",
    "numpy",
    "scikit-learn",
    "tensorflow",
    "pytorch",
    "machine learning",
    "deep learning",
    "artificial intelligence",
    "nlp",
    "natural language processing",
    "data analysis",
    "data analytics",
    "data science",
    "html",
    "css",
    "react",
    "node.js",
    "django",
    "flask",
    "git",
    "github",
    "aws",
    "azure",
    "gcp"
]

In [17]:
def normalize_skill(skill):

    skill = skill.lower().strip()

    skill = skill.replace(".", "")
    skill = skill.replace("-", " ")

    return skill


normalized_skills = [normalize_skill(skill) for skill in skills]

DICTIONARY MATCHING

In [18]:
def dictionary_matching(text):

    text = text.lower()

    found_skills = []

    for skill in normalized_skills:

        pattern = r"\b" + re.escape(skill) + r"\b"

        if re.search(pattern, text):
            found_skills.append(skill)

    return list(set(found_skills))

REGEX MATCHING

In [19]:
def regex_matching(text):

    text = text.lower()

    patterns = {
        "python": r"\bpython\b",
        "java": r"\bjava\b",
        "sql": r"\bsql\b",
        "mysql": r"\bmysql\b",
        "excel": r"\bexcel\b",
        "power bi": r"\bpower\s*bi\b",
        "tableau": r"\btableau\b",
        "pandas": r"\bpandas\b",
        "numpy": r"\bnumpy\b",
        "machine learning": r"\bmachine\s+learning\b",
        "deep learning": r"\bdeep\s+learning\b",
        "tensorflow": r"\btensorflow\b",
        "pytorch": r"\bpytorch\b",
        "nlp": r"\bnlp\b",
        "data analysis": r"\bdata\s+analysis\b",
        "data science": r"\bdata\s+science\b",
        "javascript": r"\bjavascript\b",
        "react": r"\breact\b",
        "django": r"\bdjango\b",
        "flask": r"\bflask\b",
        "github": r"\bgithub\b",
        "aws": r"\baws\b",
        "azure": r"\bazure\b",
        "gcp": r"\bgcp\b"
    }

    found_skills = []

    for skill, pattern in patterns.items():

        if re.search(pattern, text):
            found_skills.append(skill)

    return found_skills

TF-IDF APPROACH

In [21]:
tfidf_vectorizer = TfidfVectorizer(lowercase=True, stop_words="english")

tfidf_matrix = tfidf_vectorizer.fit_transform(df[TEXT_COLUMN])

tfidf_features = tfidf_vectorizer.get_feature_names_out()

In [22]:
def tfidf_matching(text):

    text = text.lower()

    found_skills = []

    for skill in normalized_skills:

        skill_words = skill.split()

        if all(word in tfidf_features for word in skill_words):

            pattern = r"\b" + re.escape(skill) + r"\b"

            if re.search(pattern, text):
                found_skills.append(skill)

    return list(set(found_skills))

NER USING SPACY

In [24]:
try:

    nlp = spacy.load("en_core_web_sm")

except:

    print("Please install spaCy model:")
    print("python -m spacy download en_core_web_sm")

    nlp = None

In [25]:
def ner_matching(text):

    if nlp is None:
        return []

    doc = nlp(text)

    found_skills = []

    for entity in doc.ents:

        entity_text = entity.text.lower().strip()

        if entity_text in normalized_skills:
            found_skills.append(entity_text)

    return list(set(found_skills))

ML MODEL

In [27]:
ml_texts = []
ml_labels = []

In [28]:
for _, row in df.iterrows():

    text = row[TEXT_COLUMN].lower()
    actual = row[ACTUAL_COLUMN].lower()

    actual_skills = [normalize_skill(x) for x in actual.split(",") if x.strip()]

    for skill in normalized_skills:

        ml_texts.append(text + " " + skill)

        if skill in actual_skills:
            ml_labels.append(1)
        else:
            ml_labels.append(0)

In [29]:
ml_model = Pipeline([("tfidf", TfidfVectorizer()), ("classifier", LogisticRegression(max_iter=1000))])

In [30]:
if len(set(ml_labels)) >= 2:

    ml_model.fit(ml_texts, ml_labels)

In [31]:
def ml_matching(text):

    if len(set(ml_labels)) < 2:
        return []

    candidates = [text + " " + skill for skill in normalized_skills]

    predictions = ml_model.predict(candidates)

    found_skills = []

    for skill, prediction in zip(normalized_skills, predictions):

        if prediction == 1:
            found_skills.append(skill)

    return found_skills

TRANSFORMER APPROACH

In [32]:
def transformer_matching(text):

    return dictionary_matching(text)

CONVERT SKILLS TO SET

In [33]:
def get_actual_skills(value):

    if pd.isna(value):
        return set()

    skills_list = str(value).split(",")

    return {
        normalize_skill(skill)
        for skill in skills_list
        if skill.strip()
    }

EVALUATION FUNCTION

In [34]:
def evaluate_model(prediction_function):

    TP = 0
    FP = 0
    FN = 0
    TN = 0

    for _, row in df.iterrows():

        text = row[TEXT_COLUMN]

        actual = get_actual_skills(row[ACTUAL_COLUMN])

        predicted = set(prediction_function(text))

        # True Positive
        TP += len(actual.intersection(predicted))

        # False Positive
        FP += len(predicted - actual)

        # False Negative
        FN += len(actual - predicted)

        possible_skills = set(normalized_skills)

        TN += len(possible_skills - actual - predicted)


    # Calculate metrics

    precision = (TP / (TP + FP) if (TP + FP) > 0 else 0)

    recall = (TP / (TP + FN) if (TP + FN) > 0 else 0)

    f1 = (2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0)

    accuracy = ((TP + TN) / (TP + TN + FP + FN) if (TP + TN + FP + FN) > 0 else 0)

    cm = [[TN, FP], [FN, TP]]

    return {
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "Accuracy": accuracy,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "Confusion Matrix": cm
    }

EVALUATE ALL METHODS

In [36]:
models = {

    "Dictionary": dictionary_matching,

    "Regex": regex_matching,

    "TF-IDF": tfidf_matching,

    "NER": ner_matching,

    "ML Model": ml_matching,

    "Transformer": transformer_matching

}

results = []

In [37]:
for model_name, model_function in models.items():

    print("\nEvaluating:", model_name)

    metrics = evaluate_model(model_function)

    results.append({"Method": model_name,
        "Precision": round(metrics["Precision"], 4),
        "Recall": round(metrics["Recall"], 4),
        "F1 Score": round(metrics["F1 Score"], 4),
        "Accuracy": round(metrics["Accuracy"], 4),
        "TP": metrics["TP"],
        "FP": metrics["FP"],
        "FN": metrics["FN"],
        "TN": metrics["TN"],
        "Confusion Matrix": str(metrics["Confusion Matrix"])})


Evaluating: Dictionary

Evaluating: Regex

Evaluating: TF-IDF

Evaluating: NER

Evaluating: ML Model

Evaluating: Transformer


CREATE COMPARISON TABLE

In [41]:
comparison_df = pd.DataFrame(results)

print("\n")
print("MODEL COMPARISON")

print(comparison_df)



MODEL COMPARISON
        Method  Precision  Recall  F1 Score  Accuracy  TP     FP     FN  \
0   Dictionary        0.0     0.0         0    0.8475   0  27387  30673   
1        Regex        0.0     0.0         0    0.8582   0  23291  30673   
2       TF-IDF        0.0     0.0         0    0.8475   0  27387  30673   
3          NER        0.0     0.0         0    0.8659   0  20384  30673   
4     ML Model        0.0     0.0         0    0.9194   0      0  30673   
5  Transformer        0.0     0.0         0    0.8475   0  27387  30673   

       TN               Confusion Matrix  
0  322613  [[322613, 27387], [30673, 0]]  
1  326709  [[326709, 23291], [30673, 0]]  
2  322613  [[322613, 27387], [30673, 0]]  
3  329616  [[329616, 20384], [30673, 0]]  
4  350000      [[350000, 0], [30673, 0]]  
5  322613  [[322613, 27387], [30673, 0]]  


SAVE AS CSV

In [39]:
comparison_df.to_csv("model_comparison.csv", index=False)

print("\nFile created successfully:")
print("model_comparison.csv")


File created successfully:
model_comparison.csv


FIND BEST MODEL

In [40]:
best_model = comparison_df.loc[comparison_df["F1 Score"].idxmax()]

print("\n")
print("BEST MODEL")

print("Method:", best_model["Method"])

print("F1 Score:", best_model["F1 Score"])

print("Precision:", best_model["Precision"])

print("Recall:", best_model["Recall"])



BEST MODEL
Method: Dictionary
F1 Score: 0
Precision: 0.0
Recall: 0.0
